##### Q4 - Data Integration Module
##### Join tables using documented keys and verify row counts

In [1]:
import pandas as pd
import logging
from pathlib import Path
from typing import Dict, Tuple, Optional

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [3]:
def load_cleaned_data(processed_dir: Path) -> Dict[str, pd.DataFrame]:
    """
    Load all cleaned CSV files from processed directory
    
    Args:
        processed_dir: Path to processed data directory
    
    Returns:
        Dictionary of dataframes
    """
    logger.info(f"Loading cleaned data from: {processed_dir}")
    
    cleaned_data = {}
    
    files = {
        'sales': 'sales_cleaned.csv',
        'orders': 'orders_cleaned.csv',
        'inventory': 'inventory_cleaned.csv',
        'warehouses': 'warehouses_cleaned.csv'
    }
    
    for name, filename in files.items():
        filepath = processed_dir / filename
        if filepath.exists():
            df = pd.read_csv(filepath)
            cleaned_data[name] = df
            logger.info(f"  - Loaded {name}: {len(df)} rows")
        else:
            logger.warning(f"  - File not found: {filepath}")
    
    return cleaned_data

In [4]:
def verify_join(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    left_on: str,
    right_on: str,
    join_type: str = 'left',
    suffixes: Tuple[str, str] = ('_left', '_right')
) -> Tuple[pd.DataFrame, dict]:
    """
    Perform join and verify row counts, with unmatched key analysis
    """
    # Store original sizes
    left_rows = len(left_df)
    right_rows = len(right_df)
    
    # Identify unmatched keys before join
    left_keys = set(left_df[left_on].dropna())
    right_keys = set(right_df[right_on].dropna())
    
    unmatched_left = left_keys - right_keys
    unmatched_right = right_keys - left_keys
    matched_keys = left_keys & right_keys
    
    # Perform join with custom suffixes
    if join_type == 'inner':
        joined = pd.merge(left_df, right_df, 
                         left_on=left_on, right_on=right_on, 
                         how='inner', suffixes=suffixes)
    elif join_type == 'left':
        joined = pd.merge(left_df, right_df, 
                         left_on=left_on, right_on=right_on, 
                         how='left', suffixes=suffixes)
    elif join_type == 'right':
        joined = pd.merge(left_df, right_df, 
                         left_on=left_on, right_on=right_on, 
                         how='right', suffixes=suffixes)
    elif join_type == 'outer':
        joined = pd.merge(left_df, right_df, 
                         left_on=left_on, right_on=right_on, 
                         how='outer', suffixes=suffixes)
    else:
        raise ValueError(f"Unknown join type: {join_type}")
    
    # Convert set to list for slicing
    unmatched_left_list = list(unmatched_left) if unmatched_left else []
    unmatched_right_list = list(unmatched_right) if unmatched_right else []
    
    # Calculate statistics
    stats = {
        'join_type': join_type,
        'left_on': left_on,
        'right_on': right_on,
        'left_rows': left_rows,
        'right_rows': right_rows,
        'joined_rows': len(joined),
        'matched_keys': len(matched_keys),
        'unmatched_left_keys': len(unmatched_left),
        'unmatched_right_keys': len(unmatched_right),
        'unmatched_left_examples': unmatched_left_list[:5],
        'unmatched_right_examples': unmatched_right_list[:5],
        'row_count_delta': len(joined) - left_rows if join_type == 'left' else 
                          len(joined) - right_rows if join_type == 'right' else
                          len(joined) - max(left_rows, right_rows)
    }
    
    # Log results
    logger.info(f"\n  Join Statistics ({join_type} join):")
    logger.info(f"    Left: {left_rows} rows, Right: {right_rows} rows")
    logger.info(f"    Joined: {len(joined)} rows")
    logger.info(f"    Matched keys: {len(matched_keys)}")
    logger.info(f"    Unmatched in left: {len(unmatched_left)}")
    logger.info(f"    Unmatched in right: {len(unmatched_right)}")
    
    if unmatched_left:
        logger.info(f"    Unmatched left examples: {unmatched_left_list[:3]}")
    if unmatched_right:
        logger.info(f"    Unmatched right examples: {unmatched_right_list[:3]}")
    
    return joined, stats

In [5]:
def integrate_sales_orders(sales: pd.DataFrame, orders: pd.DataFrame) -> Tuple[pd.DataFrame, dict]:
    """
    Integrate sales and orders data on SKU and warehouse
    """
    logger.info("\n" + "="*60)
    logger.info("INTEGRATING SALES & ORDERS")
    logger.info("="*60)
    
    # Make copies to avoid modifying originals
    sales_copy = sales.copy()
    orders_copy = orders.copy()
    
    # Create composite keys for join
    sales_copy['sku_wh'] = sales_copy['sku'].astype(str) + '_' + sales_copy['warehouse_id'].astype(str)
    orders_copy['sku_wh'] = orders_copy['sku'].astype(str) + '_' + orders_copy['warehouse_id'].astype(str)
    
    # Perform join with suffixes
    integrated, stats = verify_join(
        left_df=sales_copy,
        right_df=orders_copy,
        left_on='sku_wh',
        right_on='sku_wh',
        join_type='outer',
        suffixes=('_sales', '_orders')  # مشخص کردن پسوندها
    )
    
    # Drop the composite key
    integrated = integrated.drop(columns=['sku_wh'], errors='ignore')
    
    logger.info(f"\n  ✅ Sales-Orders integration complete: {len(integrated)} rows")
    logger.info(f"  Columns after integration: {integrated.columns.tolist()}")
    
    return integrated, stats

In [6]:
def integrate_inventory(inventory: pd.DataFrame, 
                        integrated_df: pd.DataFrame) -> Tuple[pd.DataFrame, dict]:
    """
    Integrate inventory data with existing integrated data
    """
    logger.info("\n" + "="*60)
    logger.info("INTEGRATING INVENTORY")
    logger.info("="*60)
    
    # Make copies
    inventory_copy = inventory.copy()
    integrated_copy = integrated_df.copy()
    
    # شناسایی نام ستون‌های sku و warehouse_id در integrated_copy
    sku_col = None
    wh_col = None
    
    for col in integrated_copy.columns:
        if col.lower() in ['sku', 'sku_sales', 'sku_orders', 'sku_left', 'sku_right']:
            sku_col = col
        if col.lower() in ['warehouse_id', 'warehouse_id_sales', 'warehouse_id_orders', 
                          'warehouse_id_left', 'warehouse_id_right']:
            wh_col = col
    
    # اگر ستون‌های مورد نظر پیدا نشدند، از اولین ستون‌های موجود استفاده کن
    if sku_col is None:
        # پیدا کردن ستونی که شبیه sku است
        for col in integrated_copy.columns:
            if 'sku' in col.lower():
                sku_col = col
                break
    
    if wh_col is None:
        # پیدا کردن ستونی که شبیه warehouse_id است
        for col in integrated_copy.columns:
            if 'warehouse' in col.lower() or 'wh' in col.lower():
                wh_col = col
                break
    
    # اگر هنوز پیدا نشد، از ستون‌های اصلی استفاده کن
    if sku_col is None:
        sku_col = 'sku'
    if wh_col is None:
        wh_col = 'warehouse_id'
    
    logger.info(f"  Using columns: sku='{sku_col}', warehouse_id='{wh_col}'")
    
    # Create composite keys
    inventory_copy['sku_wh'] = inventory_copy['sku'].astype(str) + '_' + inventory_copy['warehouse_id'].astype(str)
    integrated_copy['sku_wh'] = integrated_copy[sku_col].astype(str) + '_' + integrated_copy[wh_col].astype(str)
    
    # Perform join
    integrated, stats = verify_join(
        left_df=integrated_copy,
        right_df=inventory_copy,
        left_on='sku_wh',
        right_on='sku_wh',
        join_type='left',
        suffixes=('', '_inv')
    )
    
    # Drop composite key
    integrated = integrated.drop(columns=['sku_wh'], errors='ignore')
    
    logger.info(f"\n  ✅ Inventory integration complete: {len(integrated)} rows")
    logger.info(f"  Columns after integration: {integrated.columns.tolist()}")
    
    return integrated, stats

In [7]:
def integrate_warehouses(warehouses: pd.DataFrame,
                         integrated_df: pd.DataFrame) -> Tuple[pd.DataFrame, dict]:
    """
    Integrate warehouse master data with integrated data
    """
    logger.info("\n" + "="*60)
    logger.info("INTEGRATING WAREHOUSES MASTER")
    logger.info("="*60)
    
    # Make copies
    warehouses_copy = warehouses.copy()
    integrated_copy = integrated_df.copy()
    
    # شناسایی نام ستون warehouse_id در integrated_copy
    wh_col = None
    for col in integrated_copy.columns:
        if col.lower() in ['warehouse_id', 'warehouse_id_sales', 'warehouse_id_orders',
                          'warehouse_id_left', 'warehouse_id_right']:
            wh_col = col
            break
    
    if wh_col is None:
        for col in integrated_copy.columns:
            if 'warehouse' in col.lower() or 'wh' in col.lower():
                wh_col = col
                break
    
    if wh_col is None:
        wh_col = 'warehouse_id'
    
    logger.info(f"  Using warehouse_id column: '{wh_col}'")
    
    # Perform join on warehouse_id
    integrated, stats = verify_join(
        left_df=integrated_copy,
        right_df=warehouses_copy,
        left_on=wh_col,
        right_on='warehouse_id',
        join_type='left',
        suffixes=('', '_wh')
    )
    
    logger.info(f"\n  ✅ Warehouse master integration complete: {len(integrated)} rows")
    logger.info(f"  Columns after integration: {integrated.columns.tolist()}")
    
    return integrated, stats

In [8]:
def run_integration_pipeline(processed_dir: Path) -> Tuple[pd.DataFrame, dict]:
    """
    Run the complete data integration pipeline
    
    Args:
        processed_dir: Directory containing cleaned CSV files
    
    Returns:
        Tuple of (master integrated dataframe, integration summary)
    """
    logger.info("="*60)
    logger.info("STARTING DATA INTEGRATION PIPELINE (Q4)")
    logger.info("="*60)
    
    # Load cleaned data
    data = load_cleaned_data(processed_dir)
    
    if not all(k in data for k in ['sales', 'orders', 'inventory', 'warehouses']):
        logger.error("Missing required data files!")
        return None, None
    
    # Step 1: Integrate Sales & Orders
    integrated, stats_sales_orders = integrate_sales_orders(
        sales=data['sales'],
        orders=data['orders']
    )
    
    # Step 2: Integrate Inventory
    integrated, stats_inventory = integrate_inventory(
        inventory=data['inventory'],
        integrated_df=integrated
    )
    
    # Step 3: Integrate Warehouses
    integrated, stats_warehouses = integrate_warehouses(
        warehouses=data['warehouses'],
        integrated_df=integrated
    )
    
    # Step 4: Clean up column names (remove suffixes from duplicate columns)
    # Keep only relevant columns
    integrated = clean_integrated_columns(integrated)
    
    # Step 5: Generate master summary
    summary = {
        'integration_date': pd.Timestamp.now().isoformat(),
        'source_files': {
            'sales': len(data['sales']),
            'orders': len(data['orders']),
            'inventory': len(data['inventory']),
            'warehouses': len(data['warehouses'])
        },
        'final_rows': len(integrated),
        'final_columns': len(integrated.columns),
        'join_statistics': {
            'sales_orders': stats_sales_orders,
            'inventory': stats_inventory,
            'warehouses': stats_warehouses
        },
        'columns': integrated.columns.tolist()
    }
    
    # Log summary
    logger.info("\n" + "="*60)
    logger.info("INTEGRATION SUMMARY")
    logger.info("="*60)
    logger.info(f"Source rows:")
    logger.info(f"  Sales: {len(data['sales'])}")
    logger.info(f"  Orders: {len(data['orders'])}")
    logger.info(f"  Inventory: {len(data['inventory'])}")
    logger.info(f"  Warehouses: {len(data['warehouses'])}")
    logger.info(f"\nFinal integrated dataset: {len(integrated)} rows, {len(integrated.columns)} columns")
    
    return integrated, summary

In [9]:
def clean_integrated_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean up column names after joins
    """
    # Create a mapping to rename columns to standard names
    rename_map = {}
    
    for col in df.columns:
        if 'sku' in col.lower() and col != 'sku':
            if not any(c for c in rename_map.values() if c == 'sku'):
                rename_map[col] = 'sku'
        
        elif 'warehouse_id' in col.lower() and col != 'warehouse_id':
            if not any(c for c in rename_map.values() if c == 'warehouse_id'):
                rename_map[col] = 'warehouse_id'
        
        elif 'on_hand' in col.lower() and col != 'on_hand_quantity':
            if not any(c for c in rename_map.values() if c == 'on_hand_quantity'):
                rename_map[col] = 'on_hand_quantity'
        
        elif 'date' in col.lower() and col != 'date':
            if not any(c for c in rename_map.values() if c == 'date'):
                rename_map[col] = 'date'
        
        elif 'order_id' in col.lower() and col != 'order_id':
            if not any(c for c in rename_map.values() if c == 'order_id'):
                rename_map[col] = 'order_id'
    
    df = df.rename(columns=rename_map)
    
    cols_to_drop = []
    for col in df.columns:
        if col.endswith('_y') or col.endswith('_right') or \
           col.endswith('_inv') or col.endswith('_wh') or \
           col.endswith('_orders') or col.endswith('_sales') or \
           col.endswith('_left') or col == 'key_0':
            cols_to_drop.append(col)
    
    df = df.drop(columns=cols_to_drop, errors='ignore')
    
    df = df.loc[:, ~df.columns.duplicated()]
    
    if 'on_hand_qty' in df.columns and 'on_hand_quantity' not in df.columns:
        df = df.rename(columns={'on_hand_qty': 'on_hand_quantity'})
    
    preferred_order = [
        'sku', 'warehouse_id', 'warehouse_name', 'city', 'region',
        'transaction_id', 'order_id', 'sales_quantity', 'ordered_quantity',
        'on_hand_quantity', 'revenue', 'price', 'order_status',
        'snapshot_date', 'date', 'order_date', 'customer_id'
    ]
    
    final_cols = [col for col in preferred_order if col in df.columns]
    
    remaining_cols = [col for col in df.columns if col not in final_cols]
    final_cols.extend(remaining_cols)
    
    df = df[final_cols]
    
    return df

In [10]:
def analyze_unmatched_keys(integrated_df: pd.DataFrame, 
                           original_data: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Analyze unmatched keys after integration
    """
    logger.info("\n" + "="*60)
    logger.info("UNMATCHED KEY ANALYSIS")
    logger.info("="*60)
    
    analysis = []
    
    inv_col = None
    for col in integrated_df.columns:
        if 'on_hand' in col.lower() or 'stock' in col.lower():
            inv_col = col
            break
    
    if inv_col is None:
        logger.warning("  No inventory column found in integrated data")
        inv_col = 'on_hand_quantity'  # fallback
    
    sku_col = None
    for col in integrated_df.columns:
        if col.lower() == 'sku':
            sku_col = col
            break
    
    if sku_col is None:
        for col in integrated_df.columns:
            if 'sku' in col.lower():
                sku_col = col
                break
    
    if sku_col is None:
        sku_col = 'sku'
    
    wh_name_col = None
    for col in integrated_df.columns:
        if 'warehouse_name' in col.lower() or 'warehouse name' in col.lower():
            wh_name_col = col
            break
    
    sales_qty_col = None
    for col in integrated_df.columns:
        if 'sales_quantity' in col.lower() or 'sales qty' in col.lower():
            sales_qty_col = col
            break
    
    orders_qty_col = None
    for col in integrated_df.columns:
        if 'ordered_quantity' in col.lower() or 'ordered qty' in col.lower() or 'order_quantity' in col.lower():
            orders_qty_col = col
            break
    
    logger.info(f"  Using columns: sku='{sku_col}', inventory='{inv_col}'")
    
    # Check for rows with missing warehouse info
    if wh_name_col:
        missing_wh = integrated_df[integrated_df[wh_name_col].isna()]
        if len(missing_wh) > 0:
            logger.info(f"⚠️  {len(missing_wh)} rows have no warehouse information")
            if sku_col in missing_wh.columns:
                sku_samples = missing_wh[sku_col].unique().tolist()[:5]
                logger.info(f"   SKUs affected: {sku_samples}...")
            analysis.append({
                'issue': 'Missing warehouse information',
                'count': len(missing_wh),
                'example_skus': missing_wh[sku_col].unique().tolist()[:5] if sku_col in missing_wh.columns else []
            })
    
    # Check for rows with missing inventory
    if inv_col in integrated_df.columns:
        missing_inv = integrated_df[integrated_df[inv_col].isna()]
        if len(missing_inv) > 0:
            logger.info(f"⚠️  {len(missing_inv)} rows have no inventory data")
            analysis.append({
                'issue': 'Missing inventory data',
                'count': len(missing_inv),
                'example_skus': missing_inv[sku_col].unique().tolist()[:5] if sku_col in missing_inv.columns else []
            })
    
    # Check for rows with no sales but orders exist
    if sales_qty_col and orders_qty_col:
        no_sales = integrated_df[
            (integrated_df[sales_qty_col].isna()) & 
            (~integrated_df[orders_qty_col].isna())
        ]
        if len(no_sales) > 0:
            logger.info(f"ℹ️  {len(no_sales)} rows have orders but no sales")
            analysis.append({
                'issue': 'Orders exist but no sales',
                'count': len(no_sales),
                'example_skus': no_sales[sku_col].unique().tolist()[:5] if sku_col in no_sales.columns else []
            })
    
    # Check for rows with no orders but sales exist
    if sales_qty_col and orders_qty_col:
        no_orders = integrated_df[
            (~integrated_df[sales_qty_col].isna()) & 
            (integrated_df[orders_qty_col].isna())
        ]
        if len(no_orders) > 0:
            logger.info(f"ℹ️  {len(no_orders)} rows have sales but no orders")
            analysis.append({
                'issue': 'Sales exist but no orders',
                'count': len(no_orders),
                'example_skus': no_orders[sku_col].unique().tolist()[:5] if sku_col in no_orders.columns else []
            })
    
    # Check for zero inventory but positive demand
    if inv_col in integrated_df.columns and orders_qty_col:
        zero_inv_demand = integrated_df[
            (integrated_df[inv_col] == 0) & 
            (~integrated_df[orders_qty_col].isna()) &
            (integrated_df[orders_qty_col] > 0)
        ]
        if len(zero_inv_demand) > 0:
            logger.info(f"⚠️  {len(zero_inv_demand)} rows have zero inventory with orders")
            analysis.append({
                'issue': 'Zero inventory with pending orders',
                'count': len(zero_inv_demand),
                'example_skus': zero_inv_demand[sku_col].unique().tolist()[:5] if sku_col in zero_inv_demand.columns else []
            })
    
    return pd.DataFrame(analysis)

In [11]:
def save_integrated_data(integrated_df: pd.DataFrame, 
                         summary: dict,
                         output_dir: Path) -> None:
    """
    Save integrated data and summary files
    """
    logger.info("\n" + "="*60)
    logger.info("SAVING INTEGRATED DATA")
    logger.info("="*60)
    
    # Create integrated directory
    integrated_dir = output_dir / 'integrated'
    integrated_dir.mkdir(parents=True, exist_ok=True)
    
    # Save main data
    output_path = integrated_dir / 'master_integrated_data.csv'
    integrated_df.to_csv(output_path, index=False)
    logger.info(f"  ✅ Saved integrated data: {output_path}")
    
    # Save summary
    import json
    summary_path = integrated_dir / 'integration_summary.json'
    with open(summary_path, 'w') as f:
        # Convert any non-serializable objects
        json_summary = json.dumps(summary, indent=2, default=str)
        f.write(json_summary)
    logger.info(f"  ✅ Saved integration summary: {summary_path}")
    
    # Save column descriptions
    col_desc = pd.DataFrame({
        'column': integrated_df.columns,
        'dtype': integrated_df.dtypes.astype(str),
        'non_null_count': integrated_df.count(),
        'null_count': integrated_df.isna().sum()
    })
    col_desc_path = integrated_dir / 'column_descriptions.csv'
    col_desc.to_csv(col_desc_path, index=False)
    logger.info(f"  ✅ Saved column descriptions: {col_desc_path}")